# Handwritten Digit Classification using CNN & RNN on MNIST Dataset

**Author:** Md. Mursalin  
**Email:** programmer.mursalin016@gmail.com

This notebook trains and evaluates both a CNN and an RNN on the MNIST handwritten digit dataset.

In [ ]:
# Install packages if needed
# !pip install torch torchvision matplotlib seaborn scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device:', device)

# Hyperparameters
batch_size = 64
learning_rate = 0.001
epochs = 5
input_size = 28
sequence_length = 28
num_classes = 10
hidden_size = 128
num_layers = 2

In [ ]:
# Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print('Training samples:', len(train_dataset))
print('Testing samples:', len(test_dataset))

In [ ]:
# CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# RNN Model
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNNModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch, 1, 28, 28) -> (batch, 28, 28)
        x = x.squeeze(1)

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        out, _ = self.rnn(x, h0)

        # last time step output
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
# Training function
def train_model(model, train_loader, criterion, optimizer, epochs, model_name='Model'):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f'{model_name} | Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%')

In [ ]:
# Evaluation function
def evaluate_model(model, test_loader, model_name='Model'):
    model.eval()

    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    accuracy = 100 * correct / total
    print(f'\n{model_name} Test Accuracy: {accuracy:.2f}%')
    print(f'\n{model_name} Classification Report:\n')
    print(classification_report(all_labels, all_preds))

    cm = confusion_matrix(all_labels, all_preds)
    return accuracy, cm

In [ ]:
# Confusion matrix plot
def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

In [ ]:
# Train and test CNN
cnn_model = CNNModel().to(device)
criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=learning_rate)

print('\nTraining CNN Model...\n')
train_model(cnn_model, train_loader, criterion, cnn_optimizer, epochs, model_name='CNN')

cnn_accuracy, cnn_cm = evaluate_model(cnn_model, test_loader, model_name='CNN')
plot_confusion_matrix(cnn_cm, title='CNN Confusion Matrix')

In [ ]:
# Train and test RNN
rnn_model = RNNModel(input_size, hidden_size, num_layers, num_classes).to(device)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

print('\nTraining RNN Model...\n')
train_model(rnn_model, train_loader, criterion, rnn_optimizer, epochs, model_name='RNN')

rnn_accuracy, rnn_cm = evaluate_model(rnn_model, test_loader, model_name='RNN')
plot_confusion_matrix(rnn_cm, title='RNN Confusion Matrix')

In [ ]:
# Final comparison
print('\n==============================')
print('Final Model Comparison')
print('==============================')
print(f'CNN Test Accuracy : {cnn_accuracy:.2f}%')
print(f'RNN Test Accuracy : {rnn_accuracy:.2f}%')

if cnn_accuracy > rnn_accuracy:
    print('CNN performed better than RNN on MNIST.')
elif rnn_accuracy > cnn_accuracy:
    print('RNN performed better than CNN on MNIST.')
else:
    print('Both models performed equally.')